# BigMart: Gamma Sales Prediction
A fresh implementation of the approach in the supplied AMBA_FINAL_2 notebook. Read VIVA_NOTES.md for explanations. This notebook includes all model code.

In [ ]:
from pathlib import Path
import os, sys, zipfile
if 'google.colab' in sys.modules:
    from google.colab import files
    if not Path('/content/BigMart_Gamma/data/train.csv').exists():
        print('Upload BigMart_Gamma.zip')
        uploaded = files.upload()
        archive = next(n for n in uploaded if n.endswith('.zip'))
        with zipfile.ZipFile(archive) as z:
            root = Path('/content').resolve()
            assert all((root / n).resolve().is_relative_to(root) for n in z.namelist())
            z.extractall(root)
    os.chdir('/content/BigMart_Gamma')
assert Path('data/train.csv').exists(), 'Run from the extracted BigMart_Gamma folder.'


## 1. Libraries and settings

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, SplineTransformer
from sklearn.linear_model import GammaRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
TARGET = 'Item_Outlet_Sales'
KEYS = ['Item_Identifier', 'Outlet_Identifier']



## 2. Build and fit the features
Store indicators, store-specific log-price effects and a product-number spline are learned on training rows only.

In [ ]:
def product_number(frame):
    # This numbering is an empirical predictor, not a physical product property.
    return frame['Item_Identifier'].str.extract(r'(\d+)$')[0].astype(float).to_numpy().reshape(-1, 1)

def encode(frame, encoder, spline):
    store = encoder.transform(frame[['Outlet_Identifier']])
    price = np.log(frame['Item_MRP'].to_numpy()).reshape(-1, 1)
    return np.column_stack([store, store * price, spline.transform(product_number(frame))])

def fit_model(frame, sales):
    # Learn every transformation from these training rows only.
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    encoder.fit(frame[['Outlet_Identifier']])
    spline = SplineTransformer(n_knots=16, degree=3)
    spline.fit(product_number(frame))
    estimator = GammaRegressor(alpha=0.0001, max_iter=5000)
    estimator.fit(encode(frame, encoder, spline), sales)
    return {'encoder': encoder, 'spline': spline, 'estimator': estimator}

def predict(bundle, frame):
    matrix = encode(frame, bundle['encoder'], bundle['spline'])
    return bundle['estimator'].predict(matrix)

def scores(actual, estimated):
    return {'RMSE': float(np.sqrt(mean_squared_error(actual, estimated))),
            'MAE': float(mean_absolute_error(actual, estimated)),
            'R2': float(r2_score(actual, estimated))}



## 3. Evaluate and generate predictions
This function checks data, performs five-fold validation, makes charts, refits all labeled rows, and saves the submission.

In [ ]:
def run():
    folder = ROOT / 'results'; folder.mkdir(exist_ok=True)
    train = pd.read_csv(ROOT / 'data/train.csv')
    test = pd.read_csv(ROOT / 'data/test.csv')
    template = pd.read_csv(ROOT / 'data/sample_submission.csv')
    for frame in (train, test):
        assert frame[KEYS + ['Item_MRP']].notna().all().all()
        assert (frame.Item_MRP > 0).all()
        assert not frame[KEYS].duplicated().any()
    assert (train[TARGET] > 0).all(), 'Gamma training requires positive sales.'
    assert template[KEYS].equals(test[KEYS])
    train.isna().sum().to_csv(folder / 'missing_values.csv')
    # Missing weight/size do not require imputation: neither is a predictor here.
    development, evaluation = train_test_split(train, test_size=.2, random_state=42)
    fold_rows = []
    for number, (a, b) in enumerate(KFold(5, shuffle=True, random_state=42).split(development), 1):
        learning, validation = development.iloc[a], development.iloc[b]
        fitted = fit_model(learning, learning[TARGET])
        fold_rows.append({'fold': number, **scores(validation[TARGET], predict(fitted, validation))})
    folds = pd.DataFrame(fold_rows)
    folds.to_csv(folder / 'cross_validation.csv', index=False)
    model = fit_model(development, development[TARGET])
    estimates = predict(model, evaluation)
    summary = {'development_rows': len(development), 'evaluation_rows': len(evaluation),
               'cv_rmse_mean': float(folds.RMSE.mean()),
               'cv_rmse_sd': float(folds.RMSE.std(ddof=0)),
               'evaluation': scores(evaluation[TARGET], estimates),
               'mean_baseline': scores(evaluation[TARGET], np.repeat(development[TARGET].mean(),len(evaluation))),
               'evaluation_note': 'Previously examined split: comparative diagnostic, not a new untouched test.'}
    details = evaluation[KEYS+[TARGET]].copy()
    details['Predicted_Sales'] = estimates
    details['Residual'] = details[TARGET] - estimates
    details.to_csv(folder / 'evaluation_predictions.csv', index=False)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].scatter(evaluation[TARGET], estimates, s=9, alpha=.25, color='teal')
    limit = max(evaluation[TARGET].max(), estimates.max())
    ax[0].plot([0,limit],[0,limit],'--',color='orange')
    ax[0].set(xlabel='Actual sales', ylabel='Predicted sales', title='Evaluation predictions')
    ax[1].scatter(estimates, details.Residual, s=9, alpha=.25,color='teal')
    ax[1].axhline(0,color='orange',linestyle='--')
    ax[1].set(xlabel='Predicted sales',ylabel='Actual - predicted',title='Residuals')
    fig.tight_layout();fig.savefig(folder/'diagnostics.png',dpi=150);plt.close(fig)
    # Final training uses all labels; competition test labels remain unavailable.
    final = fit_model(train, train[TARGET])
    submission = template.copy()
    submission[TARGET] = predict(final, test)
    assert len(submission)==len(test) and submission.columns.tolist()==KEYS+[TARGET]
    assert submission[KEYS].equals(test[KEYS])
    assert np.isfinite(submission[TARGET]).all() and (submission[TARGET]>0).all()
    submission.to_csv(folder/'gamma_submission.csv',index=False)
    joblib.dump(final, folder/'gamma_model.joblib')
    restored = joblib.load(folder/'gamma_model.joblib')
    np.testing.assert_allclose(predict(restored,test),submission[TARGET])
    (folder/'metrics.json').write_text(json.dumps(summary,indent=2))
    print(json.dumps(summary,indent=2))
    print('Created gamma_submission.csv:',submission.shape)
    return summary



## 4. Execute

In [ ]:
metrics = run()
metrics

## 5. Inspect errors and submission

In [ ]:
from IPython.display import display, Image
display(pd.read_csv(ROOT/'results/cross_validation.csv'))
display(Image(filename=str(ROOT/'results/diagnostics.png')))
display(pd.read_csv(ROOT/'results/gamma_submission.csv').head())

## 6. Predict one example

In [ ]:
saved = joblib.load(ROOT/'results/gamma_model.joblib')
row = pd.read_csv(ROOT/'data/test.csv').iloc[[0]]
display(row[KEYS+['Item_MRP']])
print('Estimated sales:', round(float(predict(saved,row)[0]),2))
if 'google.colab' in sys.modules:
    files.download(str(ROOT/'results/gamma_submission.csv'))